In [7]:
import os
import re
import sys
import importlib
import numpy as np
import sqlite3
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Ensure the workspace root is first in sys.path so Scripts catalog import resolves reliably.
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import Scripts.catalog_db as catalog_db
importlib.reload(catalog_db)
from Scripts.catalog_db import (
    init_catalog_db,
    query_product_db,
    query_products_by_region,
    query_shipping_to_region,
    simulate_order,
    get_all_skus,
    list_branches,
)

# ==============================================================================
# ⚙️ CONFIGURACIÓN DE ENTORNO Y MODELOS
# ==============================================================================
load_dotenv()

token = os.getenv("GITHUB_TOKEN")
base_url = os.getenv("OPENAI_BASE_URL")

# Modelo principal para el Agente (baja temperatura para mayor precisión comercial)
chat_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=token,
    base_url=base_url,
    temperature=0.3
)

# Modelo auxiliar para resúmenes de memoria larga
summary_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=token,
    base_url=base_url,
    temperature=0
)

# Embeddings para tu base de conocimiento fija (RAG)
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=token,
    base_url=base_url
)


# ==============================================================================
# 📖 BASE DE CONOCIMIENTO (REQUISITOS MÍNIMOS ESTÁTICOS)
# ==============================================================================
documento = """
Notebook ideal para programación y gaming:
- Mínimo 16GB RAM
- SSD 512GB
- GPU dedicada (RTX 3050 o superior)
- Procesador i5 o Ryzen 5 hacia arriba

En Chile (PC Factory):
- Gama media: 600.000 - 900.000 CLP
- Gama alta: 900.000 - 1.500.000 CLP

Marcas recomendadas:
- ASUS TUF
- Acer Nitro
- Lenovo Legion
"""

def dividir_texto(texto, size=200):
    return [texto[i:i+size] for i in range(0, len(texto), size)]

chunks = dividir_texto(documento)
vector_db = []

for chunk in chunks:
    emb = embeddings_model.embed_query(chunk)
    vector_db.append({"texto": chunk, "embedding": emb})


def similitud(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def obtener_contexto_estatico(pregunta, top_k=2):
    emb_pregunta = embeddings_model.embed_query(pregunta)
    resultados = []
    for item in vector_db:
        score = similitud(emb_pregunta, item["embedding"])
        resultados.append((score, item["texto"]))
    resultados.sort(reverse=True)
    return "\n".join([texto for _, texto in resultados[:top_k]])

# Iniciar base de datos SQLite del catálogo
init_catalog_db()

# ==============================================================================
# 🛠️ AGENTE: HERRAMIENTA AUTOMATIZADA DE CONSULTA DE PRODUCTO
# ==============================================================================

def _format_currency(value: int) -> str:
    return f"${value:,} CLP"


def _format_block(lines: list) -> str:
    return "\n".join(lines)


def _find_first_keyword(text: str, candidates: list) -> str | None:
    normalized = text.upper()
    for candidate in candidates:
        if candidate.upper() in normalized:
            return candidate
    return None


@tool
def consultar_producto_avanzado(termino_busqueda: str) -> str:
    """
    Busca automáticamente un producto en la base de datos usando su SKU o palabras clave de su nombre.
    Devuelve stock, precio de lista, descuentos y precios finales.
    """
    query = termino_busqueda.strip()
    if not query:
        return "Por favor, ingresa un término de búsqueda válido (SKU o nombre de componente)."

    if any(word in query.lower() for word in ["región", "region", "comuna", "envío", "envio", "sucursal"]):
        return consultar_envios_region(query)

    sku, producto = query_product_db(query)
    if not producto:
        return f"No encontré ningún componente o periférico con el término '{termino_busqueda}' en la base de datos."

    nombre = producto["nombre"]
    precio_lista = producto["precio_lista"]
    desc_porcentaje = producto["descuento_efectivo"]
    stock_actual = producto["total_stock"]
    inventario = producto.get("inventory", [])

    ahorro = int(precio_lista * desc_porcentaje)
    precio_efectivo = precio_lista - ahorro

    if stock_actual == 0:
        status_stock = "❌ AGOTADO TEMPORALMENTE. No es posible procesar pedidos de esta pieza."
    elif stock_actual <= 3:
        status_stock = f"⚠️ ¡ÚLTIMAS UNIDADES EN TIENDA! Quedan solo {stock_actual} unidades."
    else:
        status_stock = f"✅ Stock Disponible ({stock_actual} unidades)."

    inventario_texto = "\n".join(
        f"  - {item['branch_codigo']} ({item['branch_nombre']}): {item['cantidad']} unidades"
        for item in inventario
    ) if inventario else "No hay sucursales con stock registrado para este producto."

    lines = [
        f"Resultado de búsqueda ({sku}):",
        f"- Componente: {nombre}",
        f"- Estado: {status_stock}",
        f"- Precio con Tarjetas/Crédito: {_format_currency(precio_lista)}",
    ]

    if desc_porcentaje > 0:
        lines.extend([
            f"- Descuento Efectivo/Transferencia: {int(desc_porcentaje * 100)}%",
            f"- Precio Final Efectivo: {_format_currency(precio_efectivo)} (Ahorras {_format_currency(ahorro)})",
        ])
    else:
        lines.append("- Producto sin descuentos promocionales adicionales en este momento.")

    lines.append("- Inventario por sucursal:")
    lines.append(inventario_texto)

    return _format_block(lines)


@tool
def consultar_envios_region(region_o_comuna: str) -> str:
    """
    Consulta qué sucursales y stock hay disponibles para una región o comuna.
    """
    query = region_o_comuna.strip()
    if not query:
        return "Por favor, especifica una región o comuna para consultar envíos."

    resultados = query_shipping_to_region(query)
    if not resultados:
        return f"No encontré envíos ni stock disponible para la región/comuna '{query}'."

    lines = [
        f"Disponibilidad de envíos para '{query}':",
    ]
    for row in resultados:
        lines.append(
            f"- Sucursal {row['branch_codigo']} ({row['branch_nombre']}) — {row['region']} / {row['comuna']}: "
            f"{row['stock_total']} unidades disponibles en {row['productos_disponibles']} productos."
        )

    lines.append("Puedes solicitar un simulacro de compra usando la herramienta 'simular_compra'.")
    return _format_block(lines)


@tool
def simular_compra(orden_texto: str) -> str:
    """
    Simula un pedido a partir de un texto libre que incluya sucursal, SKU y cantidad.
    """
    texto = orden_texto.strip()
    if not texto:
        return "Por favor, describe el pedido con sucursal, SKU y cantidad."

    branch_candidates = [branch["codigo"] for branch in list_branches()]
    sku_candidates = get_all_skus()
    branch_code = _find_first_keyword(texto, branch_candidates)
    sku = _find_first_keyword(texto, sku_candidates)
    cantidad_match = re.search(r"(\d+)", texto)
    cliente_match = re.search(r"cliente[:\s]+([A-Za-zñÑáéíóúÁÉÍÓÚ ]+)", texto, re.IGNORECASE)
    cantidad = int(cantidad_match.group(1)) if cantidad_match else None
    cliente = cliente_match.group(1).strip() if cliente_match else None

    if not branch_code or not sku or not cantidad:
        return (
            "No pude identificar completamente la orden. Asegúrate de incluir:"
            " sucursal (código), SKU y cantidad. Ejemplo: 'Simular compra en SCL-CENTRO SKU GPU-NVIDIA-RTX4070-SUPER cantidad 2'."
        )

    try:
        resultado = simulate_order(branch_code, [(sku, cantidad)], cliente_nombre=cliente)
    except ValueError as e:
        return str(e)

    lines = [
        f"Simulación de compra para sucursal {resultado['branch']['codigo']} ({resultado['branch']['branch_nombre']}):",
        f"- Cliente: {resultado['cliente_nombre'] or 'No especificado'}",
    ]

    for item in resultado['items']:
        lines.append(
            f"- {item['cantidad']} x {item['sku']} ({item['nombre']}) @ {_format_currency(item['precio_unitario'])} "
            f"cada uno | Subtotal: {_format_currency(item['subtotal'])} | Disponible: {item['disponible']}"
        )

    lines.append(f"Total estimado: {_format_currency(resultado['total'])}")
    lines.append("Esta es una simulación. Si deseas confirmar el pedido, puedes solicitarlo como una orden real.")
    return _format_block(lines)


# Lista unificada de herramientas del Agente
tools = [consultar_producto_avanzado, consultar_envios_region, simular_compra]

# PROMPT DEL SISTEMA: Modificador de conducta para el agente ReAct
prompt_sistema = """Eres el asistente virtual avanzado y asesor experto de PC Factory Chile.
Tu misión es guiar al usuario en la compra de hardware, periféricos o armados de PC.
- Cuando el usuario pregunte por precios, stock, piezas específicas o si hay ofertas, debes usar SIEMPRE la herramienta 'consultar_producto_avanzado'.
- Cuando el usuario pregunte por envíos, disponibilidad regional o stock por región/comuna, usa la herramienta 'consultar_envios_region'.
- Si el usuario desea saber si puede comprar un producto, emplea la herramienta 'simular_compra' para validar stock y mostrar el total estimado.
- Si el producto tiene descuento, resalta de forma entusiasta el 'Precio Final Efectivo'.
- Si el producto está agotado o quedan pocas unidades, adviértele al cliente de forma proactiva.
- Responde siempre en Pesos Chilenos con formato legible (ej: $369,990 CLP).
- Usa un formato estructurado en tus explicaciones si te piden recomendaciones complejas de hardware.
"""

# Inicializamos el Agente ReAct con LangGraph
app_agente = create_react_agent(chat_llm, tools, prompt=prompt_sistema)

# ==============================================================================
# 🧠 MEMORIA Y RESUMEN AUTOMÁTICO
# ==============================================================================
store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


def auto_summarize(session_id: str, max_messages=6):
    history = get_session_history(session_id)
    if len(history.messages) > max_messages:
        messages_to_summarize = history.messages[:-2]
        conversation_text = ""
        for msg in messages_to_summarize:
            role = "Usuario" if msg.type == "human" else "Asistente"
            conversation_text += f"{role}: {msg.content}\n"
        
        summary_prompt = f"Resume brevemente los requisitos del hardware, piezas consultadas y presupuesto: {conversation_text}"
        summary_response = summary_llm.invoke(summary_prompt)
        
        recent_messages = history.messages[-2:]
        history.clear()
        history.add_ai_message(f"[RESUMEN DE CONVERSACIÓN ANTERIOR]: {summary_response.content}")
        history.messages.extend(recent_messages)

# ==============================================================================
# 🚀 FLUJO DE EJECUCIÓN CONTINUA
# ==============================================================================
def ejecutar_chatbot_agente():
    session_id = "sesion_agente_pcfactory_final"
    history = get_session_history(session_id)

    inputs = [
        "Hola, quiero armar un PC gamer y tengo dudas de qué piezas tienen disponibles.",
        "¿Tienen stock del procesador Ryzen 7800X3D o el i7 14700K? ¿Cuál me conviene?",
        "Espectacular. ¿Y a cuánto tienen la tarjeta de video RTX 4070 Super? ¿Tiene algún descuento en efectivo?"
    ]

    for user_input in inputs:
        print(f"\n➡️ Usuario: {user_input}")
        auto_summarize(session_id)
        contexto_estatico = obtener_contexto_estatico(user_input)
        input_enriquecido = f"Contexto Base de Requisitos:\n{contexto_estatico}\n\nConsulta del Usuario: {user_input}"
        history.add_user_message(input_enriquecido)
        try:
            response = app_agente.invoke(
                {"messages": history.messages},
                config={"configurable": {"thread_id": session_id}}
            )
            respuesta_final = response["messages"][-1].content
            print(f"🤖 Agente PC Factory:\n{respuesta_final}")
            history.add_ai_message(respuesta_final)
        except Exception as e:
            print(f"❌ Error en la ejecución del agente: {e}")
        print("-" * 50)

if __name__ == "__main__":
    ejecutar_chatbot_agente()


C:\Users\diego\AppData\Local\Temp\ipykernel_10324\953701467.py:279: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app_agente = create_react_agent(chat_llm, tools, prompt=prompt_sistema)



➡️ Usuario: Hola, quiero armar un PC gamer y tengo dudas de qué piezas tienen disponibles.
🤖 Agente PC Factory:
¡Hola! Estoy aquí para ayudarte a armar tu PC gamer. Para ofrecerte las mejores opciones, necesito saber un poco más sobre tus preferencias. Aquí tienes algunas preguntas que me ayudarán a guiarte:

1. **Presupuesto**: ¿Cuál es tu rango de precio para el armado del PC?
2. **Uso principal**: ¿Te enfocarás más en gaming, programación o ambos?
3. **Componentes específicos**: ¿Tienes alguna preferencia por marcas o componentes específicos (como procesador, GPU, etc.)?
4. **Almacenamiento**: ¿Prefieres un SSD, HDD o ambos?

Con esta información, podré buscar las mejores opciones disponibles en PC Factory.
--------------------------------------------------

➡️ Usuario: ¿Tienen stock del procesador Ryzen 7800X3D o el i7 14700K? ¿Cuál me conviene?
🤖 Agente PC Factory:
Lamentablemente, no encontré stock disponible para el procesador **Ryzen 7800X3D** ni para el **i7 14700K** en la ba